In [1]:
# Step 1: Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

# Step 2: Load Heart Disease dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]
data = pd.read_csv(url, names=columns, na_values='?')
print("Dataset Loaded Successfully")
print(data.head())

# Step 3: Remove missing values
data = data.dropna()

# Step 4: Select important columns
data = data[['age', 'cp', 'thalach', 'target']].copy()

# Convert target into binary values
data['target'] = (data['target'] > 0).astype(int)

# Step 5: Convert continuous values into categories
data['age'], age_bins = pd.cut(
    data['age'], bins=3, labels=False,
    retbins=True, include_lowest=True
)

data['thalach'], thalach_bins = pd.cut(
    data['thalach'], bins=3, labels=False,
    retbins=True, include_lowest=True
)

# Step 6: Split dataset into training and testing data
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data['target']
)

# Step 7: Create Bayesian Network
model = DiscreteBayesianNetwork([
    ('age', 'target'),
    ('cp', 'target'),
    ('thalach', 'target')
])

# Step 8: Train the Bayesian Network
model.fit(train_data)

# Step 9: Perform inference
inference = VariableElimination(model)
predictions = []
for _, row in test_data.iterrows():
    result = inference.map_query(
        variables=['target'],
        evidence={
            'age': int(row['age']),
            'cp': int(row['cp']),
            'thalach': int(row['thalach'])
        },
        show_progress=False
    )
    predictions.append(result['target'])

# Step 10: Calculate accuracy
accuracy = accuracy_score(
    test_data['target'], predictions
)
print("\nModel Accuracy:", accuracy * 100, "%")
print("\nClassification Report:")
print(classification_report(test_data['target'], predictions))

# Step 11: Predict for a new patient
new_age = 55
new_cp = 2
new_thalach = 150
patient = {
    'age': int(pd.cut(
        [new_age], bins=age_bins,
        labels=False, include_lowest=True
    )[0]),

    'cp': new_cp,

    'thalach': int(pd.cut(
        [new_thalach], bins=thalach_bins,
        labels=False, include_lowest=True
    )[0])
}

result = inference.map_query(
    variables=['target'],
    evidence=patient,
    show_progress=False
)


# Step 12: Display prediction
print("\nPatient Prediction:")

if result['target'] == 1:
    print("Heart Disease Detected")
else:
    print("No Heart Disease Detected")

/Users/musharrafkolhar/college_practicals/ir_practicals/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/musharrafkolhar/college_practicals/ir_practicals/venv/lib/python3.14/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Dataset Loaded Successfully
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  target  
0    3.0  0.0   6.0       0  
1    2.0  3.0   3.0       2  
2    2.0  2.0   7.0       1  
3    3.0  0.0   3.0       0  
4    1.0  0.0   3.0       0  

Model Accuracy: 71.66666666666667 %

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.75      0.74        32
           1       0.70      0.68      0.69        28

    accuracy                           0.72        60
   macro avg       0.72      0.71      0.71       